# 02 Synthetic Error Insertion on FinQA (First 5 Rows)

All config lives in `config.py`.
All Groq logic lives in `groq_utils.py`.


In [11]:
# %pip install -q datasets groq python-dotenv

In [12]:
import sys, re
sys.path.insert(0, "..")

from datasets import load_dataset
import groq_utils as gu
from config import FINQA_SUBSET, RAGBENCH_REPO

API_KEY = gu.load_api_key()
MODEL   = gu.pick_model(API_KEY)
client  = gu.make_client(API_KEY)

N_ROWS  = 5

.env  : c:\Users\Akash\Desktop\FRED\.env
Model : openai/gpt-oss-20b


In [13]:
ds    = load_dataset(RAGBENCH_REPO, FINQA_SUBSET, trust_remote_code=True)
train = ds["train"]
print(f"FinQA train: {len(train):,} rows | columns: {train.column_names}")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'rungalileo/ragbench' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


FinQA train: 12,502 rows | columns: ['id', 'question', 'documents', 'response', 'generation_model_name', 'annotating_model_name', 'dataset_name', 'documents_sentences', 'response_sentences', 'sentence_support_information', 'unsupported_response_sentence_keys', 'adherence_score', 'overall_supported_explanation', 'relevance_explanation', 'all_relevant_sentence_keys', 'all_utilized_sentence_keys', 'trulens_groundedness', 'trulens_context_relevance', 'ragas_faithfulness', 'ragas_context_relevance', 'gpt3_adherence', 'gpt3_context_relevance', 'gpt35_utilization', 'relevance_score', 'utilization_score', 'completeness_score']


In [14]:
results = []

for idx in range(N_ROWS):
    row = train[idx]
    documents = row.get("documents") or row.get("document") or row.get("context") or ""
    question  = row.get("question")  or row.get("query")    or ""
    response  = row.get("response")  or row.get("answer")   or row.get("output") or ""

    print(f"\nRow {idx+1} of {N_ROWS}\n")
    print(f"Question:\n{question}\n")
    print(f"Original Response:\n{response}\n")

    corrupted = gu.insert_error(client, MODEL, documents, question, response)
    print(f"Corrupted Response (tagged):\n{corrupted}\n")

    results.append({"idx": idx, "question": question, "response": response, "corrupted": corrupted})

print(f"Done: {len(results)} rows processed.")


Row 1 of 5

Question:
what is the yearly amortization rate related to the trademarks?

Original Response:
The yearly amortization rate related to trademarks is 8 years.

Corrupted Response (tagged):
The yearly amortization rate related to trademarks is <NUMERIC><delete>8 years</delete><mark>10 years</mark></NUMERIC>.


Row 2 of 5

Question:
was initial health care trend rate higher in 2017 than 2016?

Original Response:
No, the initial health care trend rate was higher in 2016 than in 2017. In 2016, the initial health care trend rate was 8.25%, while in 2017, it was 8.00%.

Corrupted Response (tagged):
No, the initial health care trend rate was higher in 2016 than in 2017. In 2016, the initial health care trend rate was <NUMERICAL><delete>8.25%</delete><mark>7.25%</mark></NUMERICAL>, while in 2017, it was 8.00%.


Row 3 of 5

Question:
what is the average expected volatility for the years 2007-2009?

Original Response:
The average expected volatility for the years 2007-2009 is as foll

In [15]:
TAG_PATTERN = re.compile(r"<(temporal|numerical|entity|relation|contradictory|unverifiable)>", re.IGNORECASE)

print(f"{'Row':<5} {'Tag found':<15} {'Error type'}")
for r in results:
    match = TAG_PATTERN.search(r["corrupted"])
    etype = match.group(1) if match else "NONE"
    status = "OK" if match else "FAIL"
    print(f"{r['idx']+1:<5} {status:<15} {etype}")

Row   Tag found       Error type
1     FAIL            NONE
2     OK              NUMERICAL
3     FAIL            NONE
4     OK              NUMERICAL
5     OK              NUMERICAL
